# H03 — 计时的时候，按钮还要响应

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H03 — 计时的时候，按钮还要响应。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H03-timing.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

给提醒器加一段倒计时。等待五秒的代码很简单，但五秒内还能取消吗？用观测选择主循环、Timer 和 Interrupt。

前置：H02 持续工程，STEP 1 时间/消抖测试。

## 本次够用的知识

esp_timer_get_time() 返回启动以来的微秒。非阻塞状态机用截止时间，而不是等待整段倒计时。默认 esp_timer callback 在共享 Timer task 中执行，回调应短；GPIO ISR 只能使用适用的 FromISR API，不能打印、分配或做长工作。Interrupt 会更快看见边沿，但不会自动消抖。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 保留原有按钮与灯 · Guided

保存 H02 起点，记录现有 tick 周期和按键行为。明确本课不替换 Controller。先预测：在 app_main 每轮 vTaskDelay(5000 ms)，取消请求要等多久？
GPIO19/18 的接线保持不变。可以先复用 STEP 1 虚拟时钟测试，再测真实输入。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## Timer 只送通知 · Guided

读计时示例，main 的 REQUIRES 增加 esp_timer。首次可在 Git 暂存点上跑独立示例，再恢复并把 timer 初始化与通知处理增量合并；owner 必须在创建 timer 前设置。
每个通知记录 dt_us 与 pending，不用一个日志就宣称精度。采集十次，比较间隔、累积漂移、任务忙时的 pending。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H03/examples/timer_event.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_timer esp_driver_gpio freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include <cinttypes>
#include "esp_timer.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

static TaskHandle_t owner = nullptr;
static void on_timer(void*) {
    // Default esp_timer callback runs in a task, not in an ISR.
    xTaskNotifyGive(owner);
}

extern "C" void app_main() {
    owner = xTaskGetCurrentTaskHandle();
    esp_timer_create_args_t args{};
    args.callback = on_timer;
    args.name = "heartbeat";
    esp_timer_handle_t timer = nullptr;
    ESP_ERROR_CHECK(esp_timer_create(&args, &timer));
    ESP_ERROR_CHECK(esp_timer_start_periodic(timer, 1000000));
    int64_t previous = esp_timer_get_time();
    while (true) {
        const uint32_t pending = ulTaskNotifyTake(pdTRUE, pdMS_TO_TICKS(20));
        if (pending > 0) {
            const int64_t now = esp_timer_get_time();
            ESP_LOGI("timer", "pending=%" PRIu32 " dt_us=%" PRId64, pending,
                     now - previous);
            previous = now;
        }
        // Insert one short Controller tick here; never wait for the whole countdown.
    }
}


## 测量响应而不是猜 · Modify

在 Controller 增加五秒倒计时，按钮再次按下取消。用 esp_timer_get_time 的截止时间检查，在等待通知的短超时后处理输入。
至少测十次“开始后马上取消”，记录你定义的响应要求和实际范围；串口打印也会影响观测，不把日志时间戳当示波器。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 边沿发生后谁来处理 · Guided

另读 ISR 示例。主任务接通知后读取电平，ISR 只告知发生边沿。暂时比较轮询与通知，不把两个 app_main 同时登记。
ISR 使用 vTaskNotifyGiveFromISR；默认 Timer task callback 使用 xTaskNotifyGive。观察一次物理按压可能产生多个 edges，解释为何业务事件数不能直接等于边沿数。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H03/examples/interrupt.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_timer esp_driver_gpio freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "driver/gpio.h"
#include "esp_attr.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

static TaskHandle_t owner;
static void IRAM_ATTR on_edge(void*) {
    BaseType_t woken = pdFALSE;
    vTaskNotifyGiveFromISR(owner, &woken);
    if (woken) {
        portYIELD_FROM_ISR();
    }
}

extern "C" void app_main() {
    owner = xTaskGetCurrentTaskHandle();
    gpio_config_t input{};
    input.pin_bit_mask = 1ULL << GPIO_NUM_19;
    input.mode = GPIO_MODE_INPUT;
    input.pull_up_en = GPIO_PULLUP_ENABLE;
    input.intr_type = GPIO_INTR_ANYEDGE;
    ESP_ERROR_CHECK(gpio_config(&input));
    // No ESP_INTR_FLAG_IRAM: handler annotation alone does not make the entire call path IRAM-safe.
    ESP_ERROR_CHECK(gpio_install_isr_service(0));
    ESP_ERROR_CHECK(gpio_isr_handler_add(GPIO_NUM_19, on_edge, nullptr));
    while (true) {
        const uint32_t edges = ulTaskNotifyTake(pdTRUE, pdMS_TO_TICKS(1000));
        if (edges > 0) {
            ESP_LOGI("edge", "edges=%lu raw=%d", (unsigned long)edges,
                     gpio_get_level(GPIO_NUM_19));
        }
    }
}


## 把两秒误写成两毫秒 · Debug

把 Timer 周期 1000000 暂时改成 1000，观察日志积压或 pending；恢复。读单位，而不是用增加任务优先级掩盖问题。
watchdog 实验采用日志诊断：核对 reset reason、阻塞任务与 idle 饥饿。不要先关闭 watchdog。若准备制造忙循环，先保留可重新烧录固件和复位路径，agent 应分步骤指导。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 稳定按键 + 可取消计时 · Build

需求：稳定状态持续 30 ms 才采纳；长按不重复触发；上电按住不得自动启动；计时期间可取消；Timer/ISR 回调不修改复杂业务对象。
实现自己的事件输入与状态机；用 fake 时间测试 29/30 ms 边界、连续抖动、截止点取消，再在板上测。比较轮询方案是否已足够，不强制使用 Interrupt。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

可选自动验收：按 `step2/tests/README.md` 的接口契约，显式指定自己的 header 路径运行 CMake/CTest。默认测维护者参考，不能计作自己的代码通过。


## 选择最小机制 · Open

写下为何选 polling、timer 或 interrupt，以及哪些状态只由一个执行上下文写入。展示故障日志和定位过程。
H04 会增加连续输入，保留短 tick 与响应预算。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 计时期间能取消且有实际响应记录
- [ ] 抖动与单次事件分别验收
- [ ] Timer task / ISR 的 API 和职责区分清楚

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H03',
    [
        '计时期间能取消且有实际响应记录',
        '抖动与单次事件分别验收',
        'Timer task / ISR 的 API 和职责区分清楚',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H03/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/system/esp_timer.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/system/esp_timer.html)
- [api-reference/system/wdts.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/system/wdts.html)